# V_ADD_STUDENT_DEGREE_STATUS unresolved-case quarantine v1

This notebook audits the known unresolved suspicious records connected to `V_ADD_STUDENT_DEGREE_STATUS` and related course-attempt review examples. The purpose is quarantine and traceability only: unresolved rows are isolated from analytical/training datasets, while raw values are preserved in audit outputs for later manual review.

No recommendation logic, model training, or production feature engineering is performed here.

## Safety notes

- This notebook does not connect to any database.
- This notebook does not search for raw data files.
- The only files read directly from the project root are the small review CSVs: `test.csv`, `test1.csv`, `test2.csv`, `test3.csv`, and `decribe.csv`.
- Full status/course data paths are optional and must be set manually in the configuration cell.
- Raw input files are never modified.
- Rows are quarantined into audit outputs instead of permanently deleted.

## Imports and configuration

In [7]:
from pathlib import Path
import re

from src.paths import RAW_DIR, ensure_parent
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 160)

# Manual full-data inputs. Keep these as None unless you intentionally set an external path.
# Supported formats: .csv, .parquet, .xlsx, .xls
STATUS_INPUT_PATH = RAW_DIR / "v_add_student_degree_status.parquet"
COURSE_INPUT_PATH = RAW_DIR / "v_crg_student_course_raw.parquet"

QUARANTINE_VERSION = "v1"

# Hard-scan thresholds. These should stay conservative until domain owners approve changes.
MAX_REASONABLE_SEMESTER_COURSES = 10
MAX_REASONABLE_SEMESTER_CREDITS = 35
MAX_REASONABLE_TOTAL_FAIL_CREDITS = 250
GPA_PERCENT_MIN = 0
GPA_PERCENT_MAX = 100
GPA_POINTS_MIN = 0
GPA_POINTS_MAX = 4

ROOT_REVIEW_FILE_NAMES = ["test.csv", "test1.csv", "test2.csv", "test3.csv", "decribe.csv"]

# This uses the current working directory, or the known notebook-folder relationship.
# It does not search arbitrary parent folders for raw data.
start_dir = Path.cwd().resolve()
PROJECT_ROOT = start_dir
if not all((PROJECT_ROOT / file_name).exists() for file_name in ROOT_REVIEW_FILE_NAMES):
    known_notebook_folder = start_dir.name == "V_ADD_STUDENT_DEGREE_STATUS" and start_dir.parent.name == "notebooks"
    notebook_project_root = start_dir.parent.parent if known_notebook_folder else start_dir
    if all((notebook_project_root / file_name).exists() for file_name in ROOT_REVIEW_FILE_NAMES):
        PROJECT_ROOT = notebook_project_root

AUDIT_OUTPUT_DIR = PROJECT_ROOT / "data" / "audit" / "V_ADD_STUDENT_DEGREE_STATUS"
AUDIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Audit output directory: {AUDIT_OUTPUT_DIR}")

Project root: D:\AI\Real projects\Academic_Advisor
Audit output directory: D:\AI\Real projects\Academic_Advisor\data\audit\V_ADD_STUDENT_DEGREE_STATUS


## Load root review CSV files

Only the explicitly listed root review CSVs are loaded in this section.

In [8]:
review_paths = {file_name: PROJECT_ROOT / file_name for file_name in ROOT_REVIEW_FILE_NAMES}
missing_review_files = [file_name for file_name, path in review_paths.items() if not path.exists()]
assert not missing_review_files, f"Missing expected root review CSVs: {missing_review_files}"


def read_review_csv(path: Path) -> pd.DataFrame:
    """Read one of the small root review CSVs without mutating raw values."""
    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp1256")


review_frames = {file_name: read_review_csv(path) for file_name, path in review_paths.items()}

for file_name, frame in review_frames.items():
    display(Markdown(f"### `{file_name}` ({len(frame):,} rows, {len(frame.columns):,} columns)"))
    display(frame.head(30))

### `test.csv` (17 rows, 17 columns)

,Unnamed: 0,student_course_id,student_id,course_id,part_id,grade_id,final_mark,points,finish_status,course_name_sl,register_status,study_mode,degree_id,degree_name_sl,faculty_id,course_credits,active
0,161024,1045272.111,3472.111,715.111,20164.0,985.111,30.0,0.0,F,أمراض الأذن والأنف والحنجرة,E,C,2.111,دكتور في الطب,2.111,3.0,A
1,161026,1045274.111,3472.111,689.111,20164.0,985.111,29.0,0.0,F,الأحياء الدقيقة (1),E,C,2.111,دكتور في الطب,2.111,4.0,A
2,161030,1045278.111,3472.111,690.111,20164.0,985.111,17.0,0.0,F,الأحياء الدقيقة (2),E,C,2.111,دكتور في الطب,2.111,4.0,A
3,161031,1045279.111,3472.111,712.111,20164.0,985.111,49.0,0.0,F,الجراحة (5),E,C,2.111,دكتور في الطب,2.111,6.0,A
4,161032,1045280.111,3472.111,692.111,20164.0,985.111,41.0,0.0,F,الصحة العامة والمهنية,E,C,2.111,دكتور في الطب,2.111,3.0,A
5,161033,1045281.111,3472.111,697.111,20164.0,985.111,39.0,0.0,F,الطب الباطني (3),E,C,2.111,دكتور في الطب,2.111,6.0,A
6,168769,1049771.111,3472.111,676.111,20164.0,985.111,31.0,0.0,F,الكيمياء الحيوية الطبية (1),E,C,2.111,دكتور في الطب,2.111,5.0,A
7,176154,1048683.111,3472.111,704.111,20164.0,985.111,37.0,0.0,F,أمراض الأطفال (1),E,C,2.111,دكتور في الطب,2.111,4.5,A
8,176155,1048684.111,3472.111,702.111,20164.0,985.111,40.0,0.0,F,أمراض الجلد والأمراض المنتقلة بالجنس,E,C,2.111,دكتور في الطب,2.111,3.0,A
9,176156,1048685.111,3472.111,711.111,20164.0,985.111,45.0,0.0,F,الجراحة (4),E,C,2.111,دكتور في الطب,2.111,5.0,A


### `test1.csv` (11 rows, 17 columns)

,Unnamed: 0,student_course_id,student_id,course_id,part_id,grade_id,final_mark,points,finish_status,course_name_sl,register_status,study_mode,degree_id,degree_name_sl,faculty_id,course_credits,active
0,657580,2364195.111,11015.111,659.111,20222.0,NaN,NaN,NaN,NaN,علم الحياة والخلية,D,C,2.111,دكتور في الطب,2.111,4.0,A
1,660281,2357368.111,11015.111,673.111,20222.0,981.111,67.0,2.25,P,التشريح (2),R,C,2.111,دكتور في الطب,2.111,4.0,A
2,660282,2357369.111,11015.111,674.111,20222.0,980.111,73.0,2.50,P,التشريح (3),R,C,2.111,دكتور في الطب,2.111,4.0,A
3,660283,2357370.111,11015.111,683.111,20222.0,978.111,81.0,3.00,P,علم الأدوية (2) (الخاص),R,C,2.111,دكتور في الطب,2.111,4.0,A
4,687297,2366827.111,11015.111,659.111,20222.0,983.111,55.0,1.75,P,علم الحياة والخلية,R,C,2.111,دكتور في الطب,2.111,4.0,A
5,688322,2365374.111,11015.111,658.111,20222.0,983.111,56.0,1.75,P,الكيمياء العامة والعضوية,R,C,2.111,دكتور في الطب,2.111,4.0,A
6,692712,2365389.111,11015.111,955.111,20222.0,NaN,NaN,NaN,NaN,اللغة العربية(1),D,C,2.111,دكتور في الطب,2.111,2.0,A
7,723125,2364829.111,11015.111,660.111,20222.0,NaN,NaN,NaN,NaN,الوراثة,D,C,2.111,دكتور في الطب,2.111,2.0,A
8,729404,2365763.111,11015.111,706.111,20222.0,982.111,60.0,2.00,P,أمراض الأطفال (2),R,C,2.111,دكتور في الطب,2.111,5.0,A
9,729405,2365764.111,11015.111,660.111,20222.0,NaN,NaN,NaN,NaN,الوراثة,D,C,2.111,دكتور في الطب,2.111,2.0,A


### `test2.csv` (33 rows, 17 columns)

,Unnamed: 0,student_course_id,student_id,course_id,part_id,grade_id,final_mark,points,finish_status,course_name_sl,register_status,study_mode,degree_id,degree_name_sl,faculty_id,course_credits,active
0,132382,988015.111,3470.111,699.111,20161.0,985.111,42.0,0.0,F,الطب الباطني (5),R,C,2.111,دكتور في الطب,2.111,6.0,A
1,132383,988016.111,3470.111,698.111,20161.0,985.111,34.0,0.0,F,الطب الباطني (4),R,C,2.111,دكتور في الطب,2.111,6.0,A
2,137076,988115.111,3470.111,711.111,20161.0,985.111,45.0,0.0,F,الجراحة (4),R,C,2.111,دكتور في الطب,2.111,5.0,A
3,154623,1048134.111,3470.111,711.111,20164.0,985.111,42.0,0.0,F,الجراحة (4),E,C,2.111,دكتور في الطب,2.111,5.0,A
4,154624,1048135.111,3470.111,698.111,20164.0,985.111,43.0,0.0,F,الطب الباطني (4),E,C,2.111,دكتور في الطب,2.111,6.0,A
5,154625,1048136.111,3470.111,699.111,20164.0,985.111,45.0,0.0,F,الطب الباطني (5),E,C,2.111,دكتور في الطب,2.111,6.0,A
6,154626,1048138.111,3470.111,665.111,20164.0,985.111,42.0,0.0,F,علم المصطلحات الطبية,E,C,2.111,دكتور في الطب,2.111,2.0,A
7,188099,1039920.111,3470.111,665.111,20163.0,985.111,32.0,0.0,F,علم المصطلحات الطبية,R,C,2.111,دكتور في الطب,2.111,2.0,A
8,232269,1082977.111,3470.111,704.111,20171.0,985.111,47.0,0.0,F,أمراض الأطفال (1),R,C,2.111,دكتور في الطب,2.111,4.5,A
9,241454,1082883.111,3470.111,709.111,20171.0,985.111,33.0,0.0,F,جراحة (2),R,C,2.111,دكتور في الطب,2.111,6.0,A


### `test3.csv` (17 rows, 17 columns)

,Unnamed: 0,student_course_id,student_id,course_id,part_id,grade_id,final_mark,points,finish_status,course_name_sl,register_status,study_mode,degree_id,degree_name_sl,faculty_id,course_credits,active
0,161024,1045272.111,3472.111,715.111,20164.0,985.111,30.0,0.0,F,أمراض الأذن والأنف والحنجرة,E,C,2.111,دكتور في الطب,2.111,3.0,A
1,161026,1045274.111,3472.111,689.111,20164.0,985.111,29.0,0.0,F,الأحياء الدقيقة (1),E,C,2.111,دكتور في الطب,2.111,4.0,A
2,161030,1045278.111,3472.111,690.111,20164.0,985.111,17.0,0.0,F,الأحياء الدقيقة (2),E,C,2.111,دكتور في الطب,2.111,4.0,A
3,161031,1045279.111,3472.111,712.111,20164.0,985.111,49.0,0.0,F,الجراحة (5),E,C,2.111,دكتور في الطب,2.111,6.0,A
4,161032,1045280.111,3472.111,692.111,20164.0,985.111,41.0,0.0,F,الصحة العامة والمهنية,E,C,2.111,دكتور في الطب,2.111,3.0,A
5,161033,1045281.111,3472.111,697.111,20164.0,985.111,39.0,0.0,F,الطب الباطني (3),E,C,2.111,دكتور في الطب,2.111,6.0,A
6,168769,1049771.111,3472.111,676.111,20164.0,985.111,31.0,0.0,F,الكيمياء الحيوية الطبية (1),E,C,2.111,دكتور في الطب,2.111,5.0,A
7,176154,1048683.111,3472.111,704.111,20164.0,985.111,37.0,0.0,F,أمراض الأطفال (1),E,C,2.111,دكتور في الطب,2.111,4.5,A
8,176155,1048684.111,3472.111,702.111,20164.0,985.111,40.0,0.0,F,أمراض الجلد والأمراض المنتقلة بالجنس,E,C,2.111,دكتور في الطب,2.111,3.0,A
9,176156,1048685.111,3472.111,711.111,20164.0,985.111,45.0,0.0,F,الجراحة (4),E,C,2.111,دكتور في الطب,2.111,5.0,A


### `decribe.csv` (8 rows, 26 columns)

,Unnamed: 0,student_status_id,student_id,request_id,part_id,degree_id,gpa_percent,gpa_points,start_part_id,finish_part_id,start_agpa_percent,start_agpa_points,end_agpa_percent,end_agpa_points,semester_reg_courses,semester_reg_credits,semester_pass_courses,semester_pass_credits,semester_fail_courses,semester_fail_credits,total_pass_courses,total_pass_credits,total_fail_courses,total_fail_credits,reg_total_semesters,start_level_id
0,count,154779.000000,154779.000000,146661.000000,154779.000000,152653.000000,154768.000000,154768.000000,154774.000000,107879.000000,154768.000000,154768.000000,154768.000000,154768.000000,154768.000000,154768.000000,154768.000000,154768.000000,154768.000000,154768.000000,154768.000000,154768.000000,154768.000000,154768.000000,154162.000000,152660.000000
1,mean,278818.125847,12619.951120,343303.190604,20192.225308,7.731702,56.519266,1.808421,20167.986445,20207.811326,59.206310,1.947342,64.134566,2.115224,4.589146,12.939535,3.557363,10.027305,0.652544,1.861309,27.673744,78.400102,7.142568,20.660363,8.235421,475.624003
2,std,70557.472276,6133.201133,224592.506891,35.019862,7.166146,26.146758,1.052316,33.068373,30.030378,22.860714,0.877536,16.142495,0.706645,2.378229,6.693185,2.469030,6.977219,1.186610,3.504629,21.029037,62.213661,11.772033,36.086469,5.993601,219.168329
3,min,113278.111000,48.111000,7157.111000,20051.000000,1.111000,0.000000,0.000000,20051.000000,20061.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.111000
4,25%,236841.611000,9365.111000,146497.111000,20164.000000,2.111000,50.000000,1.080000,20141.000000,20191.000000,58.410000,1.720000,60.940000,1.890000,3.000000,8.000000,1.000000,4.000000,0.000000,0.000000,10.000000,26.000000,0.000000,0.000000,3.000000,291.111000
5,50%,286820.111000,11940.111000,271619.111000,20201.000000,3.111000,63.940000,2.050000,20171.000000,20213.000000,64.940000,2.130000,65.980000,2.190000,5.000000,15.000000,4.000000,10.000000,0.000000,0.000000,25.000000,68.000000,3.000000,8.000000,7.000000,582.111000
6,75%,330925.611000,14938.111000,538037.111000,20221.000000,13.111000,74.000000,2.610000,20201.000000,20232.000000,71.630000,2.490000,72.580000,2.540000,6.000000,18.000000,6.000000,16.000000,1.000000,3.000000,43.000000,122.000000,9.000000,26.000000,12.000000,631.111000
7,max,403812.111000,28890.111000,844441.000000,20254.000000,24.111000,100.000000,4.000000,20242.000000,20251.000000,99.780000,4.000000,99.780000,4.000000,17.000000,72.500000,13.000000,49.000000,17.000000,66.500000,106.000000,334.000000,193.000000,713.500000,49.000000,862.111000


## Summarize review cases

The summary below separates row-level example files from the `decribe.csv` describe table. It is intentionally conservative: the notebook records the IDs present in the review examples, but only the manually defined unresolved cases are turned into quarantine rules.

In [9]:
def detect_review_kind(frame: pd.DataFrame) -> str:
    if frame.empty or len(frame.columns) == 0:
        return "empty"
    first_col = frame.columns[0]
    first_values = set(frame[first_col].astype("string").str.lower().dropna().head(12).tolist())
    describe_markers = {"count", "mean", "std", "min", "25%", "50%", "75%", "max"}
    if first_values.intersection(describe_markers):
        return "describe_summary"
    return "row_examples"


def compact_unique_values(frame: pd.DataFrame, column: str, max_values: int = 12) -> str:
    if column not in frame.columns:
        return ""
    values = frame[column].dropna().astype("string").drop_duplicates().sort_values().tolist()
    if len(values) <= max_values:
        return ", ".join(values)
    shown_values = ", ".join(values[:max_values])
    return f"{shown_values} ... (+{len(values) - max_values} more)"


def describe_max_value(frame: pd.DataFrame, column: str) -> float:
    if column not in frame.columns or frame.empty:
        return np.nan
    first_col = frame.columns[0]
    stat_labels = frame[first_col].astype("string").str.lower()
    max_rows = frame.loc[stat_labels.eq("max"), column]
    if max_rows.empty:
        return np.nan
    return pd.to_numeric(max_rows.iloc[0], errors="coerce")


def summarize_review_frame(file_name: str, frame: pd.DataFrame) -> dict:
    kind = detect_review_kind(frame)
    duplicate_student_part_course_keys = 0
    if kind == "row_examples" and {"student_id", "part_id", "course_id"}.issubset(frame.columns):
        key_counts = frame.groupby(["student_id", "part_id", "course_id"], dropna=False).size()
        duplicate_student_part_course_keys = int(key_counts.gt(1).sum())

    max_course_credits = np.nan
    rows_course_credits_ge_20 = 0
    if "course_credits" in frame.columns:
        course_credits = pd.to_numeric(frame["course_credits"], errors="coerce")
        max_course_credits = course_credits.max()
        rows_course_credits_ge_20 = int(course_credits.ge(20).sum())

    return {
        "source_file": file_name,
        "review_kind": kind,
        "rows": len(frame),
        "columns": len(frame.columns),
        "student_ids_seen": compact_unique_values(frame, "student_id") if kind == "row_examples" else "",
        "part_ids_seen": compact_unique_values(frame, "part_id") if kind == "row_examples" else "",
        "course_ids_seen": compact_unique_values(frame, "course_id") if kind == "row_examples" else "",
        "max_course_credits": max_course_credits,
        "rows_course_credits_ge_20": rows_course_credits_ge_20,
        "duplicate_student_part_course_keys": duplicate_student_part_course_keys,
        "describe_max_semester_reg_courses": describe_max_value(frame, "semester_reg_courses") if kind == "describe_summary" else np.nan,
        "describe_max_semester_reg_credits": describe_max_value(frame, "semester_reg_credits") if kind == "describe_summary" else np.nan,
        "describe_max_total_fail_credits": describe_max_value(frame, "total_fail_credits") if kind == "describe_summary" else np.nan,
    }


review_examples_summary = pd.DataFrame(
    [summarize_review_frame(file_name, frame) for file_name, frame in review_frames.items()]
)
review_examples_summary_path = AUDIT_OUTPUT_DIR / "review_examples_summary_v1.csv"
review_examples_summary.to_csv(review_examples_summary_path, index=False)

display(review_examples_summary)
print(f"Saved review summary: {review_examples_summary_path}")

,source_file,review_kind,rows,columns,student_ids_seen,part_ids_seen,course_ids_seen,max_course_credits,rows_course_credits_ge_20,duplicate_student_part_course_keys,describe_max_semester_reg_courses,describe_max_semester_reg_credits,describe_max_total_fail_credits
0,test.csv,row_examples,17,17,3472.111,20164.0,"676.111, 687.111, 689.111, 690.111, 691.111, 6...",6.0,0,1,NaN,NaN,NaN
1,test1.csv,row_examples,11,17,11015.111,20222.0,"1093.111, 658.111, 659.111, 660.111, 673.111, ...",24.0,1,2,NaN,NaN,NaN
2,test2.csv,row_examples,33,17,3470.111,"20161.0, 20163.0, 20164.0, 20171.0, 20172.0, 2...","665.111, 678.111, 689.111, 690.111, 696.111, 6...",6.0,0,0,NaN,NaN,NaN
3,test3.csv,row_examples,17,17,3472.111,20164.0,"676.111, 687.111, 689.111, 690.111, 691.111, 6...",6.0,0,1,NaN,NaN,NaN
4,decribe.csv,describe_summary,8,26,,,,NaN,0,0,17.0,72.5,713.5


Saved review summary: D:\AI\Real projects\Academic_Advisor\data\audit\V_ADD_STUDENT_DEGREE_STATUS\review_examples_summary_v1.csv


## Build quarantine rules

These rules encode only the known unresolved cases from the review files and the support-team context. They are intentionally narrow except for the `pseudo_course` rule, which isolates graduation/administrative requirements using `course_credits >= 20` or explicit name patterns.

**Warning:** The `pseudo_course` rule is global. It isolates every row with `course_credits >= 20` or a graduation/admin/national-exam course name pattern, not only the manually listed student. This is intentional for now, but should be reviewed before production use.

In [10]:
ALLOWED_ISOLATE_LEVELS = {"student_semester", "student", "course_attempt", "pseudo_course", "status_row"}
KNOWN_QUARANTINE_STUDENT_IDS = [3472.111, 11015.111, 3470.111]

quarantine_rule_records = [
    {
        "student_id": 3472.111,
        "part_id": 20164.0,
        "course_id": pd.NA,
        "isolate_level": "student_semester",
        "isolation_reason": "Unresolved student semester with around 17 course attempts and extreme registered credits; not confirmed as valid or invalid.",
        "source_file": "test.csv; test3.csv",
        "notes": "Remove matching student_id + part_id from clean analytical/training data, but keep rows in isolated audit outputs.",
        "created_in_version": QUARANTINE_VERSION,
        "student_status_id": pd.NA,
        "request_id": pd.NA,
        "degree_id": pd.NA,
    },
    {
        "student_id": 11015.111,
        "part_id": 20222.0,
        "course_id": pd.NA,
        "isolate_level": "student_semester",
        "isolation_reason": "Unresolved student semester with duplicate/unclear course-attempt rows and a graduation or administrative requirement mixed with normal courses.",
        "source_file": "test1.csv",
        "notes": "Semester-level isolation prevents unclear course attempts and status aggregates from entering analytical/training data.",
        "created_in_version": QUARANTINE_VERSION,
        "student_status_id": pd.NA,
        "request_id": pd.NA,
        "degree_id": pd.NA,
    },
    {
        "student_id": 3470.111,
        "part_id": pd.NA,
        "course_id": pd.NA,
        "isolate_level": "student",
        "isolation_reason": "Extreme accumulated failure-history pattern associated with finish_status = F; describe table shows total_fail_credits reaching 713.5.",
        "source_file": "test2.csv; decribe.csv",
        "notes": "Student-level isolation is used until this failure history is manually reconciled against cleaned course-attempt records.",
        "created_in_version": QUARANTINE_VERSION,
        "student_status_id": pd.NA,
        "request_id": pd.NA,
        "degree_id": pd.NA,
    },
    {
        "student_id": 11015.111,
        "part_id": 20222.0,
        "course_id": 1093.111,
        "isolate_level": "pseudo_course",
        "isolation_reason": "course_credits = 24 indicates a graduation/national-exam/administrative requirement rather than a normal course attempt.",
        "source_file": "test1.csv",
        "notes": "The pseudo_course rule also catches any rows with course_credits >= 20 or matching graduation/admin name patterns when applied to course-attempt data.",
        "created_in_version": QUARANTINE_VERSION,
        "student_status_id": pd.NA,
        "request_id": pd.NA,
        "degree_id": pd.NA,
    },
]

quarantine_rule_columns = [
    "student_id",
    "part_id",
    "course_id",
    "isolate_level",
    "isolation_reason",
    "source_file",
    "notes",
    "created_in_version",
    "student_status_id",
    "request_id",
    "degree_id",
]

quarantine_rules = pd.DataFrame(quarantine_rule_records).reindex(columns=quarantine_rule_columns)

assert set(quarantine_rules["isolate_level"]).issubset(ALLOWED_ISOLATE_LEVELS)
assert not quarantine_rules.duplicated(subset=["student_id", "part_id", "course_id", "isolate_level"]).any()
assert {"student_id", "part_id", "course_id", "isolate_level", "isolation_reason", "source_file", "notes", "created_in_version"}.issubset(quarantine_rules.columns)

quarantine_rules_path = AUDIT_OUTPUT_DIR / "quarantine_rules_v1.csv"
quarantine_rules.to_csv(quarantine_rules_path, index=False)

display(quarantine_rules)
print(f"Saved quarantine rules: {quarantine_rules_path}")

,student_id,part_id,course_id,isolate_level,isolation_reason,source_file,notes,created_in_version,student_status_id,request_id,degree_id
0,3472.111,20164.0,<NA>,student_semester,Unresolved student semester with around 17 cou...,test.csv; test3.csv,Remove matching student_id + part_id from clea...,v1,<NA>,<NA>,<NA>
1,11015.111,20222.0,<NA>,student_semester,Unresolved student semester with duplicate/unc...,test1.csv,Semester-level isolation prevents unclear cour...,v1,<NA>,<NA>,<NA>
2,3470.111,<NA>,<NA>,student,Extreme accumulated failure-history pattern as...,test2.csv; decribe.csv,Student-level isolation is used until this fai...,v1,<NA>,<NA>,<NA>
3,11015.111,20222.0,1093.111,pseudo_course,course_credits = 24 indicates a graduation/nat...,test1.csv,The pseudo_course rule also catches any rows w...,v1,<NA>,<NA>,<NA>


Saved quarantine rules: D:\AI\Real projects\Academic_Advisor\data\audit\V_ADD_STUDENT_DEGREE_STATUS\quarantine_rules_v1.csv


## Quarantine helper functions

The functions below are generic and can be applied to status rows or course-attempt rows. Missing columns are handled safely by returning no match for that rule type.

In [ ]:
PSEUDO_COURSE_NAME_PATTERNS = [
    "graduation",
    "graduate",
    "administrative",
    "admin requirement",
    "degree requirement",
    "national exam",
    "national examination",
    "exit exam",
    "competency exam",
    "\u0627\u0645\u062a\u062d\u0627\u0646",
    "\u0627\u0644\u0627\u0645\u062a\u062d\u0627\u0646",
    "\u0648\u0637\u0646\u064a",
    "\u0645\u062a\u0637\u0644\u0628\u0627\u062a",
    "\u062a\u062e\u0631\u062c",
    "\u0627\u062f\u0627\u0631\u064a",
    "\u0625\u062f\u0627\u0631\u064a",
]


def false_mask(frame: pd.DataFrame) -> pd.Series:
    return pd.Series(False, index=frame.index, dtype=bool)


def true_mask(frame: pd.DataFrame) -> pd.Series:
    return pd.Series(True, index=frame.index, dtype=bool)


def match_scalar(frame: pd.DataFrame, column: str, value) -> pd.Series:
    if column not in frame.columns or pd.isna(value):
        return false_mask(frame)

    series = frame[column]
    numeric_series = pd.to_numeric(series, errors="coerce")
    try:
        target_value = float(value)
        values = numeric_series.to_numpy(dtype="float64")
        return pd.Series(np.isclose(values, target_value, rtol=0.0, atol=1e-9), index=frame.index)
    except (TypeError, ValueError):
        return series.astype("string").eq(str(value))


def combine_with_and(frame: pd.DataFrame, masks: list[pd.Series]) -> pd.Series:
    if not masks:
        return false_mask(frame)
    combined = true_mask(frame)
    for mask in masks:
        combined = combined & mask.fillna(False)
    return combined


def pseudo_course_mask(frame: pd.DataFrame) -> pd.Series:
    mask = false_mask(frame)

    if "course_credits" in frame.columns:
        credits = pd.to_numeric(frame["course_credits"], errors="coerce")
        mask = mask | credits.ge(20).fillna(False)

    name_columns = [column for column in frame.columns if "course_name" in column.lower() or column.lower() in {"name", "course_title"}]
    if name_columns:
        combined_names = frame.loc[:, name_columns].astype("string").fillna("").agg(" ".join, axis=1).str.lower()
        pattern = "|".join(re.escape(term.lower()) for term in PSEUDO_COURSE_NAME_PATTERNS)
        mask = mask | combined_names.str.contains(pattern, regex=True, na=False)

    return mask.fillna(False)


def build_rule_mask(frame: pd.DataFrame, rule: pd.Series) -> pd.Series:
    isolate_level = rule["isolate_level"]

    if isolate_level == "student_semester":
        return combine_with_and(frame, [match_scalar(frame, "student_id", rule["student_id"]), match_scalar(frame, "part_id", rule["part_id"])])

    if isolate_level == "student":
        return match_scalar(frame, "student_id", rule["student_id"])

    if isolate_level == "course_attempt":
        return combine_with_and(
            frame,
            [
                match_scalar(frame, "student_id", rule["student_id"]),
                match_scalar(frame, "part_id", rule["part_id"]),
                match_scalar(frame, "course_id", rule["course_id"]),
            ],
        )

    if isolate_level == "pseudo_course":
        return pseudo_course_mask(frame)

    if isolate_level == "status_row":
        candidate_keys = ["student_status_id", "student_id", "request_id", "part_id", "degree_id", "start_part_id", "finish_part_id"]
        key_masks = [match_scalar(frame, key, rule[key]) for key in candidate_keys if key in frame.columns and key in rule.index and not pd.isna(rule[key])]
        return combine_with_and(frame, key_masks)

    raise ValueError(f"Unsupported isolate_level: {isolate_level}")


def join_unique(values: pd.Series) -> str:
    unique_values = values.dropna().astype(str).drop_duplicates().tolist()
    return " | ".join(unique_values)


def apply_quarantine_rules(frame: pd.DataFrame, rules: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    helper_col = "__quarantine_row_id"
    while helper_col in frame.columns:
        helper_col = f"_{helper_col}"

    original_columns = list(frame.columns)
    working = frame.copy().assign(**{helper_col: np.arange(len(frame))})
    hit_records = []

    for rule_number, rule in rules.reset_index(drop=True).iterrows():
        mask = build_rule_mask(working, rule).fillna(False)
        matched_ids = working.loc[mask, helper_col].tolist()
        for row_id in matched_ids:
            hit_records.append(
                {
                    helper_col: row_id,
                    "matched_rule_number": rule_number,
                    "matched_isolate_level": rule["isolate_level"],
                    "matched_isolation_reason": rule["isolation_reason"],
                    "matched_source_file": rule["source_file"],
                    "matched_notes": rule["notes"],
                    "matched_created_in_version": rule["created_in_version"],
                }
            )

    hit_columns = [
        helper_col,
        "matched_rule_number",
        "matched_isolate_level",
        "matched_isolation_reason",
        "matched_source_file",
        "matched_notes",
        "matched_created_in_version",
    ]
    rule_hits = pd.DataFrame(hit_records, columns=hit_columns)

    if rule_hits.empty:
        clean = working.loc[:, original_columns].copy()
        isolated = working.iloc[0:0].loc[:, original_columns].copy()
        isolated = isolated.assign(
            isolation_level="",
            isolation_reason="",
            isolation_source_file="",
            isolation_notes="",
            isolation_version="",
        )
        return clean, isolated, rule_hits

    hit_summary = (
        rule_hits.groupby(helper_col, sort=False)
        .agg(
            isolation_level=("matched_isolate_level", join_unique),
            isolation_reason=("matched_isolation_reason", join_unique),
            isolation_source_file=("matched_source_file", join_unique),
            isolation_notes=("matched_notes", join_unique),
            isolation_version=("matched_created_in_version", join_unique),
        )
        .reset_index()
    )

    isolated_ids = set(hit_summary[helper_col].tolist())
    clean = working.loc[~working[helper_col].isin(isolated_ids), original_columns].copy()
    isolated = working.loc[working[helper_col].isin(isolated_ids), original_columns + [helper_col]].copy()
    isolated = isolated.merge(hit_summary, on=helper_col, how="left").drop(columns=[helper_col]).copy()

    return clean, isolated, rule_hits


def known_student_ids_present(frame: pd.DataFrame) -> list[float]:
    if "student_id" not in frame.columns:
        return []
    return [student_id for student_id in KNOWN_QUARANTINE_STUDENT_IDS if bool(match_scalar(frame, "student_id", student_id).any())]


def build_rule_hit_report(frame: pd.DataFrame, rule_hits: pd.DataFrame, rules: pd.DataFrame) -> pd.DataFrame:
    row_key_candidates = [
        "student_status_id",
        "student_course_id",
        "student_id",
        "part_id",
        "course_id",
        "request_id",
        "degree_id",
        "finish_status",
        "course_credits",
        "course_name_sl",
    ]
    rule_columns = [
        "student_id",
        "part_id",
        "course_id",
        "isolate_level",
        "isolation_reason",
        "source_file",
        "notes",
        "created_in_version",
    ]
    report_columns = ["quarantine_row_id", *row_key_candidates, "matched_rule_number", *[f"rule_{column}" for column in rule_columns]]

    if rule_hits.empty:
        return pd.DataFrame(columns=report_columns)

    helper_col = next(column for column in rule_hits.columns if column.endswith("quarantine_row_id"))
    row_keys = [column for column in row_key_candidates if column in frame.columns]
    row_lookup = frame.loc[:, row_keys].copy().assign(**{helper_col: np.arange(len(frame))})
    rule_lookup = rules.reset_index(drop=True).reset_index().rename(columns={"index": "matched_rule_number"})
    rule_lookup = rule_lookup.rename(columns={column: f"rule_{column}" for column in rule_columns})

    report = rule_hits.loc[:, [helper_col, "matched_rule_number"]].copy()
    report = report.merge(row_lookup, on=helper_col, how="left")
    report = report.merge(rule_lookup.loc[:, ["matched_rule_number", *[f"rule_{column}" for column in rule_columns]]], on="matched_rule_number", how="left")
    report = report.rename(columns={helper_col: "quarantine_row_id"})
    return report.copy()


review_course_examples = pd.concat(
    [frame.assign(source_file=file_name) for file_name, frame in review_frames.items() if detect_review_kind(frame) == "row_examples"],
    ignore_index=True,
    sort=False,
)
review_clean_preview, review_isolated_preview, review_rule_hits = apply_quarantine_rules(review_course_examples, quarantine_rules)

print(f"Review example rows: {len(review_course_examples):,}")
print(f"Review example rows that match quarantine rules: {len(review_isolated_preview):,}")
display(review_isolated_preview.head(30))

Review example rows: 78
Review example rows that match quarantine rules: 78


,Unnamed: 0,student_course_id,student_id,course_id,part_id,grade_id,final_mark,points,finish_status,course_name_sl,register_status,study_mode,degree_id,degree_name_sl,faculty_id,course_credits,active,source_file,isolation_level,isolation_reason,isolation_source_file,isolation_notes,isolation_version
0,161024,1045272.111,3472.111,715.111,20164.0,985.111,30.0,0.00,F,أمراض الأذن والأنف والحنجرة,E,C,2.111,دكتور في الطب,2.111,3.0,A,test.csv,student_semester,Unresolved student semester with around 17 cou...,test.csv; test3.csv,Remove matching student_id + part_id from clea...,v1
1,161026,1045274.111,3472.111,689.111,20164.0,985.111,29.0,0.00,F,الأحياء الدقيقة (1),E,C,2.111,دكتور في الطب,2.111,4.0,A,test.csv,student_semester,Unresolved student semester with around 17 cou...,test.csv; test3.csv,Remove matching student_id + part_id from clea...,v1
2,161030,1045278.111,3472.111,690.111,20164.0,985.111,17.0,0.00,F,الأحياء الدقيقة (2),E,C,2.111,دكتور في الطب,2.111,4.0,A,test.csv,student_semester,Unresolved student semester with around 17 cou...,test.csv; test3.csv,Remove matching student_id + part_id from clea...,v1
3,161031,1045279.111,3472.111,712.111,20164.0,985.111,49.0,0.00,F,الجراحة (5),E,C,2.111,دكتور في الطب,2.111,6.0,A,test.csv,student_semester,Unresolved student semester with around 17 cou...,test.csv; test3.csv,Remove matching student_id + part_id from clea...,v1
4,161032,1045280.111,3472.111,692.111,20164.0,985.111,41.0,0.00,F,الصحة العامة والمهنية,E,C,2.111,دكتور في الطب,2.111,3.0,A,test.csv,student_semester,Unresolved student semester with around 17 cou...,test.csv; test3.csv,Remove matching student_id + part_id from clea...,v1
5,161033,1045281.111,3472.111,697.111,20164.0,985.111,39.0,0.00,F,الطب الباطني (3),E,C,2.111,دكتور في الطب,2.111,6.0,A,test.csv,student_semester,Unresolved student semester with around 17 cou...,test.csv; test3.csv,Remove matching student_id + part_id from clea...,v1
6,168769,1049771.111,3472.111,676.111,20164.0,985.111,31.0,0.00,F,الكيمياء الحيوية الطبية (1),E,C,2.111,دكتور في الطب,2.111,5.0,A,test.csv,student_semester,Unresolved student semester with around 17 cou...,test.csv; test3.csv,Remove matching student_id + part_id from clea...,v1
7,176154,1048683.111,3472.111,704.111,20164.0,985.111,37.0,0.00,F,أمراض الأطفال (1),E,C,2.111,دكتور في الطب,2.111,4.5,A,test.csv,student_semester,Unresolved student semester with around 17 cou...,test.csv; test3.csv,Remove matching student_id + part_id from clea...,v1
8,176155,1048684.111,3472.111,702.111,20164.0,985.111,40.0,0.00,F,أمراض الجلد والأمراض المنتقلة بالجنس,E,C,2.111,دكتور في الطب,2.111,3.0,A,test.csv,student_semester,Unresolved student semester with around 17 cou...,test.csv; test3.csv,Remove matching student_id + part_id from clea...,v1
9,176156,1048685.111,3472.111,711.111,20164.0,985.111,45.0,0.00,F,الجراحة (4),E,C,2.111,دكتور في الطب,2.111,5.0,A,test.csv,student_semester,Unresolved student semester with around 17 cou...,test.csv; test3.csv,Remove matching student_id + part_id from clea...,v1


## Optional load full status and course-attempt dataframes

The full `V_ADD_STUDENT_DEGREE_STATUS` table is loaded only when `STATUS_INPUT_PATH` is manually set. If it remains `None` or points to a missing file, the notebook still completes the root-review audit and rule export.

In [ ]:
    صق

In [12]:
def load_table_from_path(path_value, label: str) -> pd.DataFrame | None:
    if path_value is None or str(path_value).strip() == "":
        display(Markdown(f"**{label}:** no input path provided; skipping optional load."))
        return None

    path = Path(path_value).expanduser()
    if not path.exists():
        display(Markdown(f"**{label}:** configured path does not exist, so it was skipped: `{path}`"))
        return None

    suffix = path.suffix.lower()
    if suffix == ".csv":
        frame = pd.read_csv(path)
    elif suffix == ".parquet":
        frame = pd.read_parquet(path)
    elif suffix in {".xlsx", ".xls"}:
        frame = pd.read_excel(path)
    else:
        raise ValueError(f"Unsupported file format for {label}: {path}")

    print(f"Loaded {label}: {path} ({len(frame):,} rows, {len(frame.columns):,} columns)")
    return frame


status_df = load_table_from_path(STATUS_INPUT_PATH, "V_ADD_STUDENT_DEGREE_STATUS")
course_df = load_table_from_path(COURSE_INPUT_PATH, "optional course-attempt table")

Loaded V_ADD_STUDENT_DEGREE_STATUS: D:\AI\Real projects\Academic_Advisor\data\raw\v_add_student_degree_status.parquet (154,779 rows, 29 columns)
Loaded optional course-attempt table: D:\AI\Real projects\Academic_Advisor\data\raw\v_crg_student_course_raw.parquet (965,467 rows, 16 columns)


## Describe helpers

These columns are scanned if they exist in the full status dataframe.

In [13]:
IMPORTANT_NUMERIC_COLUMNS = [
    "semester_reg_courses",
    "semester_reg_credits",
    "semester_pass_courses",
    "semester_pass_credits",
    "semester_fail_courses",
    "semester_fail_credits",
    "total_pass_courses",
    "total_pass_credits",
    "total_fail_courses",
    "total_fail_credits",
    "gpa_percent",
    "gpa_points",
    "start_agpa_percent",
    "start_agpa_points",
    "end_agpa_percent",
    "end_agpa_points",
]


def numeric_describe(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    existing_columns = [column for column in columns if column in frame.columns]
    if not existing_columns:
        return pd.DataFrame(columns=["column", "count", "mean", "std", "min", "25%", "50%", "75%", "95%", "99%", "max"])

    numeric_frame = frame.loc[:, existing_columns].apply(pd.to_numeric, errors="coerce")
    return numeric_frame.describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).T.rename_axis("column").reset_index()


def numeric_metric_snapshot(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    records = []
    for column in columns:
        if column not in frame.columns:
            continue
        values = pd.to_numeric(frame[column], errors="coerce")
        records.append(
            {
                "column": column,
                "count": int(values.count()),
                "mean": values.mean(),
                "p95": values.quantile(0.95),
                "p99": values.quantile(0.99),
                "max": values.max(),
            }
        )
    return pd.DataFrame(records)


if status_df is not None:
    describe_before = numeric_describe(status_df, IMPORTANT_NUMERIC_COLUMNS)
    describe_before_path = AUDIT_OUTPUT_DIR / "describe_before_quarantine_v1.csv"
    describe_before.to_csv(describe_before_path, index=False)
    display(describe_before)
    print(f"Saved describe-before file: {describe_before_path}")
else:
    describe_before = pd.DataFrame()
    display(Markdown("Full status data was not loaded, so before-quarantine describe output was skipped."))

,column,count,mean,std,min,25%,50%,75%,95%,99%,max
0,semester_reg_courses,154768.0,4.589146,2.378229,0.0,3.00,5.00,6.00,8.00,9.0000,17.00
1,semester_reg_credits,154768.0,12.939535,6.693185,0.0,8.00,15.00,18.00,22.50,25.0000,72.50
2,semester_pass_courses,154768.0,3.557363,2.469030,0.0,1.00,4.00,6.00,7.00,9.0000,13.00
3,semester_pass_credits,154768.0,10.027305,6.977219,0.0,4.00,10.00,16.00,21.00,24.5000,49.00
4,semester_fail_courses,154768.0,0.652544,1.186610,0.0,0.00,0.00,1.00,3.00,5.0000,17.00
5,semester_fail_credits,154768.0,1.861309,3.504629,0.0,0.00,0.00,3.00,9.00,15.0000,66.50
6,total_pass_courses,154768.0,27.673744,21.029037,0.0,10.00,25.00,43.00,65.00,77.0000,106.00
7,total_pass_credits,154768.0,78.400102,62.213661,0.0,26.00,68.00,122.00,193.00,237.0000,334.00
8,total_fail_courses,154768.0,7.142568,11.772033,0.0,0.00,3.00,9.00,30.00,53.0000,193.00
9,total_fail_credits,154768.0,20.660363,36.086469,0.0,0.00,8.00,26.00,86.00,157.0000,713.50


Saved describe-before file: D:\AI\Real projects\Academic_Advisor\data\audit\V_ADD_STUDENT_DEGREE_STATUS\describe_before_quarantine_v1.csv


## Apply quarantine

When the full status dataframe is available, matching rows are written to `isolated_status_rows_v1.parquet`, and the remaining clean rows are written to `clean_status_without_isolated_v1.parquet`.

When the full course-attempt dataframe is available, matching rows are written to `isolated_course_attempt_rows_v1.parquet`, and the remaining clean rows are written to `clean_course_attempts_without_isolated_v1.parquet`.

Rule-hit CSV files are also saved for loaded full datasets so each matched row can be traced back to the specific quarantine rule.

In [18]:
def save_parquet(frame: pd.DataFrame, path: Path) -> None:
    try:
        frame.to_parquet(path, index=False)
    except ImportError as exc:
        raise ImportError("Saving parquet requires pyarrow or fastparquet. Install one of them and rerun this cell.") from exc


def save_rule_hit_report(frame: pd.DataFrame, rule_hits: pd.DataFrame, rules: pd.DataFrame, path: Path) -> pd.DataFrame:
    report = build_rule_hit_report(frame, rule_hits, rules)
    report.to_csv(path, index=False)
    print(f"Saved quarantine rule-hit report: {path}")
    return report


if status_df is not None:
    clean_status_df, isolated_status_rows, status_rule_hits = apply_quarantine_rules(status_df, quarantine_rules)

    isolated_status_path = AUDIT_OUTPUT_DIR / "isolated_status_rows_v1.parquet"
    clean_status_path = AUDIT_OUTPUT_DIR / "clean_status_without_isolated_v1.parquet"
    status_rule_hits_path = AUDIT_OUTPUT_DIR / "status_quarantine_rule_hits_v1.csv"
    save_parquet(isolated_status_rows, isolated_status_path)
    save_parquet(clean_status_df, clean_status_path)
    status_rule_hit_report = save_rule_hit_report(status_df, status_rule_hits, quarantine_rules, status_rule_hits_path)

    assert len(clean_status_df) + len(isolated_status_rows) == len(status_df)

    print(f"Original status rows: {len(status_df):,}")
    print(f"Isolated status rows: {len(isolated_status_rows):,}")
    print(f"Clean status rows: {len(clean_status_df):,}")
    print(f"Saved isolated status rows: {isolated_status_path}")
    print(f"Saved clean status rows: {clean_status_path}")
    present_known_status_ids = known_student_ids_present(status_df)
    if len(isolated_status_rows) == 0 and present_known_status_ids:
        print(f"WARNING: Full status dataframe contains known quarantine student IDs {present_known_status_ids}, but zero status rows were isolated. Review ID formats and quarantine rule definitions.")
    display(isolated_status_rows.head(5))
else:
    clean_status_df = None
    isolated_status_rows = pd.DataFrame()
    status_rule_hits = pd.DataFrame()
    status_rule_hit_report = pd.DataFrame()
    display(Markdown("Full status data was not loaded, so status-row quarantine parquet and rule-hit outputs were skipped."))


if course_df is not None:
    clean_course_df, isolated_course_attempt_rows, course_rule_hits = apply_quarantine_rules(course_df, quarantine_rules)

    isolated_course_path = AUDIT_OUTPUT_DIR / "isolated_course_attempt_rows_v1.parquet"
    clean_course_path = AUDIT_OUTPUT_DIR / "clean_course_attempts_without_isolated_v1.parquet"
    course_rule_hits_path = AUDIT_OUTPUT_DIR / "course_quarantine_rule_hits_v1.csv"
    save_parquet(isolated_course_attempt_rows, isolated_course_path)
    save_parquet(clean_course_df, clean_course_path)
    course_rule_hit_report = save_rule_hit_report(course_df, course_rule_hits, quarantine_rules, course_rule_hits_path)

    assert len(clean_course_df) + len(isolated_course_attempt_rows) == len(course_df)

    print(f"Original course rows: {len(course_df):,}")
    print(f"Isolated course rows: {len(isolated_course_attempt_rows):,}")
    print(f"Clean course rows: {len(clean_course_df):,}")
    print(f"Saved isolated course-attempt rows: {isolated_course_path}")
    print(f"Saved clean course-attempt rows: {clean_course_path}")
    present_known_course_ids = known_student_ids_present(course_df)
    if len(isolated_course_attempt_rows) == 0 and present_known_course_ids:
        print(f"WARNING: Full course dataframe contains known quarantine student IDs {present_known_course_ids}, but zero course rows were isolated. Review ID formats and quarantine rule definitions.")
    display(isolated_course_attempt_rows.head(5))
else:
    clean_course_df = None
    isolated_course_attempt_rows = pd.DataFrame()
    course_rule_hits = pd.DataFrame()
    course_rule_hit_report = pd.DataFrame()
    display(Markdown("Full course-attempt data was not loaded, so course quarantine parquet and rule-hit outputs were skipped."))

Saved quarantine rule-hit report: D:\AI\Real projects\Academic_Advisor\data\audit\V_ADD_STUDENT_DEGREE_STATUS\status_quarantine_rule_hits_v1.csv
Original status rows: 154,779
Isolated status rows: 24
Clean status rows: 154,755
Saved isolated status rows: D:\AI\Real projects\Academic_Advisor\data\audit\V_ADD_STUDENT_DEGREE_STATUS\isolated_status_rows_v1.parquet
Saved clean status rows: D:\AI\Real projects\Academic_Advisor\data\audit\V_ADD_STUDENT_DEGREE_STATUS\clean_status_without_isolated_v1.parquet


,student_status_id,student_id,request_id,part_id,degree_id,study_mode,gpa_percent,gpa_points,start_part_id,finish_part_id,start_agpa_percent,start_agpa_points,end_agpa_percent,end_agpa_points,semester_reg_courses,semester_reg_credits,semester_pass_courses,semester_pass_credits,semester_fail_courses,semester_fail_credits,total_pass_courses,total_pass_credits,total_fail_courses,total_fail_credits,reg_total_semesters,finish_status,version_title_sl,start_level_id,start_level_name_pl,isolation_level,isolation_reason,isolation_source_file,isolation_notes,isolation_version
0,120688.111,3470.111,66714.111,20161.0,2.111,C,46.70,0.59,20161.0,20232.0,62.26,1.16,46.70,0.59,5.0,23.0,2.0,6.0,3.0,17.0,46.0,143.0,158.0,541.0,28.0,GRADUATED,القرار رقم 230,139.111,Fourth Year,student,Extreme accumulated failure-history pattern as...,test2.csv; decribe.csv,Student-level isolation is used until this fai...,v1
1,232423.111,3470.111,134004.111,20162.0,2.111,C,51.24,1.59,20161.0,20232.0,46.70,0.59,49.04,1.10,6.0,24.5,5.0,21.5,1.0,3.0,48.0,149.0,161.0,558.0,29.0,GRADUATED,القرار رقم 230,139.111,Fourth Year,student,Extreme accumulated failure-history pattern as...,test2.csv; decribe.csv,Student-level isolation is used until this fai...,v1
2,245348.111,3470.111,161428.111,20172.0,2.111,C,49.52,1.11,20161.0,20232.0,51.74,1.07,51.42,1.10,6.0,23.0,4.0,12.0,2.0,11.0,59.0,184.5,170.0,598.5,33.0,GRADUATED,القرار رقم 230,282.111,Fifth year,student,Extreme accumulated failure-history pattern as...,test2.csv; decribe.csv,Student-level isolation is used until this fai...,v1
3,234900.111,3470.111,138344.111,20163.0,2.111,C,60.67,1.78,20161.0,20232.0,49.04,1.10,50.89,1.21,4.0,9.0,3.0,7.0,1.0,2.0,53.0,170.5,162.0,561.0,30.0,GRADUATED,القرار رقم 230,282.111,Fifth year,student,Extreme accumulated failure-history pattern as...,test2.csv; decribe.csv,Student-level isolation is used until this fai...,v1
4,241184.111,3470.111,152966.111,20171.0,2.111,C,43.89,0.59,20161.0,20232.0,52.52,1.21,51.74,1.07,6.0,23.5,3.0,7.0,3.0,16.5,56.0,177.5,167.0,582.0,32.0,GRADUATED,القرار رقم 230,282.111,Fifth year,student,Extreme accumulated failure-history pattern as...,test2.csv; decribe.csv,Student-level isolation is used until this fai...,v1


Saved quarantine rule-hit report: D:\AI\Real projects\Academic_Advisor\data\audit\V_ADD_STUDENT_DEGREE_STATUS\course_quarantine_rule_hits_v1.csv
Original course rows: 965,467
Isolated course rows: 11,900
Clean course rows: 953,567
Saved isolated course-attempt rows: D:\AI\Real projects\Academic_Advisor\data\audit\V_ADD_STUDENT_DEGREE_STATUS\isolated_course_attempt_rows_v1.parquet
Saved clean course-attempt rows: D:\AI\Real projects\Academic_Advisor\data\audit\V_ADD_STUDENT_DEGREE_STATUS\clean_course_attempts_without_isolated_v1.parquet


,student_course_id,student_id,course_id,part_id,grade_id,final_mark,points,finish_status,course_name_sl,register_status,study_mode,degree_id,degree_name_sl,faculty_id,course_credits,active,isolation_level,isolation_reason,isolation_source_file,isolation_notes,isolation_version
0,433783.111,197.111,204.111,20132.0,675.111,94.0,3.5,P,اتصال اداري,R,C,19.111,إدارة الأعمال,7.111,2.0,A,pseudo_course,course_credits = 24 indicates a graduation/nat...,test1.csv,The pseudo_course rule also catches any rows w...,v1
1,453233.111,5552.111,531.111,20132.0,85.111,92.0,4.0,P,مشروع التخرج 2,R,C,3.111,هندسة البرمجيات ونظم المعلومات,5.111,3.0,A,pseudo_course,course_credits = 24 indicates a graduation/nat...,test1.csv,The pseudo_course rule also catches any rows w...,v1
2,453238.111,5617.111,531.111,20132.0,87.111,86.0,3.3,P,مشروع التخرج 2,R,C,3.111,هندسة البرمجيات ونظم المعلومات,5.111,3.0,A,pseudo_course,course_credits = 24 indicates a graduation/nat...,test1.csv,The pseudo_course rule also catches any rows w...,v1
3,453277.111,5530.111,539.111,20132.0,86.111,88.0,3.7,P,مشروع التخرج 1,R,C,3.111,هندسة البرمجيات ونظم المعلومات,5.111,3.0,A,pseudo_course,course_credits = 24 indicates a graduation/nat...,test1.csv,The pseudo_course rule also catches any rows w...,v1
4,453214.111,5505.111,531.111,20132.0,86.111,87.0,3.7,P,مشروع التخرج 2,R,C,3.111,هندسة البرمجيات ونظم المعلومات,5.111,3.0,A,pseudo_course,course_credits = 24 indicates a graduation/nat...,test1.csv,The pseudo_course rule also catches any rows w...,v1


## Describe after quarantine and compare

The comparison focuses on max, mean, p95, and p99 for important numeric status columns.

In [15]:
if clean_status_df is not None:
    describe_after = numeric_describe(clean_status_df, IMPORTANT_NUMERIC_COLUMNS)
    describe_after_path = AUDIT_OUTPUT_DIR / "describe_after_quarantine_v1.csv"
    describe_after.to_csv(describe_after_path, index=False)

    before_snapshot = numeric_metric_snapshot(status_df, IMPORTANT_NUMERIC_COLUMNS)
    after_snapshot = numeric_metric_snapshot(clean_status_df, IMPORTANT_NUMERIC_COLUMNS)
    before_after_comparison = before_snapshot.merge(after_snapshot, on="column", how="outer", suffixes=("_before", "_after"))

    display(describe_after)
    display(before_after_comparison)
    print(f"Saved describe-after file: {describe_after_path}")
else:
    describe_after = pd.DataFrame()
    before_after_comparison = pd.DataFrame()
    display(Markdown("Full status data was not loaded, so after-quarantine describe comparison was skipped."))

,column,count,mean,std,min,25%,50%,75%,95%,99%,max
0,semester_reg_courses,154744.0,4.589218,2.378107,0.0,3.00,5.00,6.00,8.00,9.000,17.00
1,semester_reg_credits,154744.0,12.938573,6.690720,0.0,8.00,15.00,18.00,22.50,25.000,53.00
2,semester_pass_courses,154744.0,3.557657,2.469008,0.0,1.00,4.00,6.00,7.00,9.000,13.00
3,semester_pass_credits,154744.0,10.027591,6.976384,0.0,4.00,10.00,16.00,21.00,24.500,47.00
4,semester_fail_courses,154744.0,0.652323,1.185986,0.0,0.00,0.00,1.00,3.00,5.000,17.00
5,semester_fail_credits,154744.0,1.860053,3.499232,0.0,0.00,0.00,3.00,9.00,15.000,53.00
6,total_pass_courses,154744.0,27.667968,21.025209,0.0,10.00,25.00,43.00,65.00,77.000,106.00
7,total_pass_credits,154744.0,78.379498,62.194884,0.0,26.00,68.00,122.00,193.00,236.500,334.00
8,total_fail_courses,154744.0,7.117465,11.588562,0.0,0.00,3.00,9.00,30.00,53.000,173.00
9,total_fail_credits,154744.0,20.568917,35.288747,0.0,0.00,8.00,26.00,86.00,155.785,664.50


,column,count_before,mean_before,p95_before,p99_before,max_before,count_after,mean_after,p95_after,p99_after,max_after
0,end_agpa_percent,154768,64.134566,83.48,89.5800,99.78,154744,64.135730,83.48,89.580,99.78
1,end_agpa_points,154768,2.115224,3.09,3.4100,4.00,154744,2.115319,3.09,3.410,4.00
2,gpa_percent,154768,56.519266,86.14,92.9066,100.00,154744,56.521101,86.14,92.900,100.00
3,gpa_points,154768,1.808421,3.25,3.5800,4.00,154744,1.808554,3.25,3.580,4.00
4,semester_fail_courses,154768,0.652544,3.00,5.0000,17.00,154744,0.652323,3.00,5.000,17.00
5,semester_fail_credits,154768,1.861309,9.00,15.0000,66.50,154744,1.860053,9.00,15.000,53.00
6,semester_pass_courses,154768,3.557363,7.00,9.0000,13.00,154744,3.557657,7.00,9.000,13.00
7,semester_pass_credits,154768,10.027305,21.00,24.5000,49.00,154744,10.027591,21.00,24.500,47.00
8,semester_reg_courses,154768,4.589146,8.00,9.0000,17.00,154744,4.589218,8.00,9.000,17.00
9,semester_reg_credits,154768,12.939535,22.50,25.0000,72.50,154744,12.938573,22.50,25.000,53.00


Saved describe-after file: D:\AI\Real projects\Academic_Advisor\data\audit\V_ADD_STUDENT_DEGREE_STATUS\describe_after_quarantine_v1.csv


## Hard scan after quarantine

If abnormal values remain after quarantine, this notebook does not silently drop more rows. It writes a remaining-suspicious report for manual review.

Important downstream note: affected aggregate features such as semester credits, pass/fail credits, GPA, and accumulated fail totals should be recalculated manually from a cleaned course-attempt table. They should not be trusted directly from the official status table while unresolved anomalies remain.

In [16]:
HARD_SCAN_FLAG_COLUMNS = [
    "flag_extreme_semester_reg_courses",
    "flag_extreme_semester_reg_credits",
    "flag_fail_credits_gt_reg_credits",
    "flag_pass_credits_gt_reg_credits",
    "flag_extreme_total_fail_credits",
    "flag_invalid_gpa_percent",
    "flag_invalid_gpa_points",
    "flag_invalid_agpa_percent",
    "flag_invalid_agpa_points",
]


def numeric_column(frame: pd.DataFrame, column: str) -> pd.Series:
    if column not in frame.columns:
        return pd.Series(np.nan, index=frame.index, dtype="float64")
    return pd.to_numeric(frame[column], errors="coerce")


def invalid_range_flag(frame: pd.DataFrame, column: str, minimum: float, maximum: float) -> pd.Series:
    values = numeric_column(frame, column)
    return values.notna() & (values.lt(minimum) | values.gt(maximum))


def add_hard_scan_flags(frame: pd.DataFrame) -> pd.DataFrame:
    semester_reg_courses = numeric_column(frame, "semester_reg_courses")
    semester_reg_credits = numeric_column(frame, "semester_reg_credits")
    semester_fail_credits = numeric_column(frame, "semester_fail_credits")
    semester_pass_credits = numeric_column(frame, "semester_pass_credits")
    total_fail_credits = numeric_column(frame, "total_fail_credits")

    flag_invalid_agpa_percent = invalid_range_flag(frame, "start_agpa_percent", GPA_PERCENT_MIN, GPA_PERCENT_MAX) | invalid_range_flag(frame, "end_agpa_percent", GPA_PERCENT_MIN, GPA_PERCENT_MAX)
    flag_invalid_agpa_points = invalid_range_flag(frame, "start_agpa_points", GPA_POINTS_MIN, GPA_POINTS_MAX) | invalid_range_flag(frame, "end_agpa_points", GPA_POINTS_MIN, GPA_POINTS_MAX)

    flags = {
        "flag_extreme_semester_reg_courses": semester_reg_courses.gt(MAX_REASONABLE_SEMESTER_COURSES).fillna(False),
        "flag_extreme_semester_reg_credits": semester_reg_credits.gt(MAX_REASONABLE_SEMESTER_CREDITS).fillna(False),
        "flag_fail_credits_gt_reg_credits": (semester_fail_credits.notna() & semester_reg_credits.notna() & semester_fail_credits.gt(semester_reg_credits)).fillna(False),
        "flag_pass_credits_gt_reg_credits": (semester_pass_credits.notna() & semester_reg_credits.notna() & semester_pass_credits.gt(semester_reg_credits)).fillna(False),
        "flag_extreme_total_fail_credits": total_fail_credits.gt(MAX_REASONABLE_TOTAL_FAIL_CREDITS).fillna(False),
        "flag_invalid_gpa_percent": invalid_range_flag(frame, "gpa_percent", GPA_PERCENT_MIN, GPA_PERCENT_MAX).fillna(False),
        "flag_invalid_gpa_points": invalid_range_flag(frame, "gpa_points", GPA_POINTS_MIN, GPA_POINTS_MAX).fillna(False),
        "flag_invalid_agpa_percent": flag_invalid_agpa_percent.fillna(False),
        "flag_invalid_agpa_points": flag_invalid_agpa_points.fillna(False),
    }

    flagged = frame.copy().assign(**flags)
    any_flag = flagged.loc[:, HARD_SCAN_FLAG_COLUMNS].any(axis=1)
    return flagged.assign(flag_any_remaining_suspicious=any_flag)


hard_scan_remaining_path = AUDIT_OUTPUT_DIR / "hard_scan_remaining_suspicious_v1.csv"

if clean_status_df is not None:
    hard_scanned_status = add_hard_scan_flags(clean_status_df)
    remaining_suspicious_rows = hard_scanned_status.loc[hard_scanned_status["flag_any_remaining_suspicious"]].copy()
    remaining_suspicious_rows.to_csv(hard_scan_remaining_path, index=False)

    flag_summary = remaining_suspicious_rows.loc[:, HARD_SCAN_FLAG_COLUMNS].sum().rename("remaining_flagged_rows").reset_index().rename(columns={"index": "flag"})
    display(flag_summary)
    display(remaining_suspicious_rows.head(50))
    print(f"Saved remaining suspicious report: {hard_scan_remaining_path}")
else:
    remaining_suspicious_rows = pd.DataFrame(columns=["note", *HARD_SCAN_FLAG_COLUMNS, "flag_any_remaining_suspicious"])
    remaining_suspicious_rows.to_csv(hard_scan_remaining_path, index=False)
    display(Markdown("Full status data was not loaded, so the remaining-suspicious report was written as an empty schema file."))
    print(f"Saved empty remaining suspicious report schema: {hard_scan_remaining_path}")

,flag,remaining_flagged_rows
0,flag_extreme_semester_reg_courses,220
1,flag_extreme_semester_reg_credits,277
2,flag_fail_credits_gt_reg_credits,45
3,flag_pass_credits_gt_reg_credits,1
4,flag_extreme_total_fail_credits,376
5,flag_invalid_gpa_percent,0
6,flag_invalid_gpa_points,0
7,flag_invalid_agpa_percent,0
8,flag_invalid_agpa_points,0


,student_status_id,student_id,request_id,part_id,degree_id,study_mode,gpa_percent,gpa_points,start_part_id,finish_part_id,start_agpa_percent,start_agpa_points,end_agpa_percent,end_agpa_points,semester_reg_courses,semester_reg_credits,semester_pass_courses,semester_pass_credits,semester_fail_courses,semester_fail_credits,total_pass_courses,total_pass_credits,total_fail_courses,total_fail_credits,reg_total_semesters,finish_status,version_title_sl,start_level_id,start_level_name_pl,flag_extreme_semester_reg_courses,flag_extreme_semester_reg_credits,flag_fail_credits_gt_reg_credits,flag_pass_credits_gt_reg_credits,flag_extreme_total_fail_credits,flag_invalid_gpa_percent,flag_invalid_gpa_points,flag_invalid_agpa_percent,flag_invalid_agpa_points,flag_any_remaining_suspicious
16,334729.111,28405.111,590060.111,20221.0,1.111,C,2.33,0.00,20221.0,20221.0,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,6.0,18.0,0.0,0.0,0.0,0.0,0.0,CANCEL_ADMISSION,القرار رقم 230,595.111,First Year,False,False,True,False,False,False,False,False,False,True
1284,204067.111,10021.111,113617.111,20161.0,13.111,C,23.80,0.19,20151.0,NaN,41.80,1.09,37.00,0.85,6.0,16.0,1.0,2.0,6.0,17.0,9.0,22.0,6.0,17.0,4.0,NaN,القرار رقم 230,170.111,Second Year,False,False,True,False,False,False,False,False,False,True
2222,168111.111,1674.111,48884.111,20123.0,13.111,C,0.00,0.00,20081.0,20161.0,26.60,0.33,26.60,0.33,17.0,53.0,0.0,0.0,17.0,53.0,5.0,17.0,33.0,104.0,8.0,CLOSE_FILE,القرار رقم 230,29.111,First Year,True,True,False,False,False,False,False,False,False,True
2224,168113.111,1674.111,48886.111,20132.0,13.111,C,0.00,0.00,20081.0,20161.0,26.60,0.33,26.60,0.33,5.0,15.0,0.0,0.0,9.0,26.0,5.0,17.0,57.0,178.0,10.0,CLOSE_FILE,القرار رقم 230,29.111,First Year,False,False,True,False,False,False,False,False,False,True
2423,167777.111,2087.111,54086.111,20143.0,13.111,C,0.00,0.00,20131.0,20153.0,39.40,0.97,36.00,0.80,3.0,7.0,0.0,0.0,4.0,9.0,8.0,19.0,19.0,49.0,6.0,FINAL_DISMISS,القرار رقم 230,170.111,Second Year,False,False,True,False,False,False,False,False,False,True
2826,242521.111,2038.111,156278.111,20172.0,13.111,C,67.60,2.38,20131.0,20173.0,63.00,2.15,64.00,2.20,11.0,21.0,9.0,19.0,2.0,2.0,57.0,157.0,10.0,26.0,14.0,GRADUATED,القرار رقم 230,313.111,Fifth year,True,False,False,False,False,False,False,False,False,True
3158,170719.111,2063.111,53862.111,20143.0,13.111,C,0.00,0.00,20131.0,20181.0,65.20,2.26,60.60,2.03,3.0,8.0,0.0,0.0,4.0,11.0,26.0,67.0,3.0,8.0,7.0,GRADUATED,القرار رقم 230,171.111,Third year,False,False,True,False,False,False,False,False,False,True
3482,148921.111,1901.111,52161.111,20152.0,13.111,C,75.26,2.65,20111.0,20152.0,71.69,2.47,72.42,2.51,11.0,23.0,11.0,23.0,0.0,0.0,57.0,155.0,6.0,11.0,11.0,GRADUATED,القرار الوزاري الموحد,33.111,Fifth year,True,False,False,False,False,False,False,False,False,True
3540,167836.111,1906.111,52216.111,20152.0,13.111,C,67.45,2.30,20111.0,20152.0,70.12,2.38,70.73,2.44,11.0,22.0,11.0,22.0,0.0,0.0,58.0,158.0,7.0,12.0,12.0,GRADUATED,القرار الوزاري الموحد,33.111,Fifth year,True,False,False,False,False,False,False,False,False,True
4395,227224.111,2431.111,131709.111,20162.0,13.111,C,46.50,1.21,20152.0,20231.0,52.98,1.26,52.70,1.52,6.0,18.0,4.0,13.0,2.0,5.0,40.0,119.0,79.0,254.0,18.0,CLOSE_FILE,القرار رقم 230,171.111,Third year,False,False,False,False,True,False,False,False,False,True


Saved remaining suspicious report: D:\AI\Real projects\Academic_Advisor\data\audit\V_ADD_STUDENT_DEGREE_STATUS\hard_scan_remaining_suspicious_v1.csv


## Final summary and next decision

This notebook creates a reproducible audit trail for unresolved suspicious cases. The quarantine rules are saved as `quarantine_rules_v1.csv` and can be reviewed before full data paths are set.

Next decision for the data owner:

1. Review `quarantine_rules_v1.csv` and confirm whether the current isolation levels are acceptable.
2. Set `STATUS_INPUT_PATH` and/or `COURSE_INPUT_PATH` manually and rerun the notebook to generate isolated and cleaned parquet files.
3. Inspect `status_quarantine_rule_hits_v1.csv` and/or `course_quarantine_rule_hits_v1.csv` when full datasets are provided.
4. Inspect `hard_scan_remaining_suspicious_v1.csv`. If rows remain, do not drop them automatically; reconcile those features against the cleaned course-attempt table and update quarantine rules in a new version.

In [17]:
print("Final quarantine audit summary")
print(f"Quarantine rules saved: {quarantine_rules_path}")
print(f"Review examples summary saved: {review_examples_summary_path}")
print(f"Remaining suspicious report saved: {hard_scan_remaining_path}")

if status_df is not None:
    print(f"Status rows: original={len(status_df):,}, isolated={len(isolated_status_rows):,}, clean={len(clean_status_df):,}")
    print(f"Status rule-hit report rows: {len(status_rule_hit_report):,}")
else:
    print("Status full-data quarantine skipped because STATUS_INPUT_PATH was not provided or was unavailable.")

if course_df is not None:
    print(f"Course rows: original={len(course_df):,}, isolated={len(isolated_course_attempt_rows):,}, clean={len(clean_course_df):,}")
    print(f"Course rule-hit report rows: {len(course_rule_hit_report):,}")
else:
    print("Course full-data quarantine skipped because COURSE_INPUT_PATH was not provided or was unavailable.")

Final quarantine audit summary
Quarantine rules saved: D:\AI\Real projects\Academic_Advisor\data\audit\V_ADD_STUDENT_DEGREE_STATUS\quarantine_rules_v1.csv
Review examples summary saved: D:\AI\Real projects\Academic_Advisor\data\audit\V_ADD_STUDENT_DEGREE_STATUS\review_examples_summary_v1.csv
Remaining suspicious report saved: D:\AI\Real projects\Academic_Advisor\data\audit\V_ADD_STUDENT_DEGREE_STATUS\hard_scan_remaining_suspicious_v1.csv
Status rows: original=154,779, isolated=24, clean=154,755
Status rule-hit report rows: 24
Course rows: original=965,467, isolated=11,900, clean=953,567
Course rule-hit report rows: 11,903
